**Random Forest_Hyperparameter_Tuning_and_Evaluation**

In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
import numpy as np

# Initialize Random Forest
rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=20,
    min_samples_split=2,
    random_state=42,
    n_jobs=-1
)

# 5-fold cross-validation on training data
cv_scores = cross_val_score(
    rf,
    X_train_ml,
    y_train.values.ravel(),
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("Individual Fold R² Scores:")
print(cv_scores)

cv_mean = cv_scores.mean()
cv_sd = cv_scores.std()

print(f"\nMean CV R² = {cv_mean:.4f}")
print(f"SD CV R²   = {cv_sd:.4f}")
print(f"RF CV R²   = {cv_mean:.4f} ± {cv_sd:.4f}")

# Train final RF model on the full training set
rf.fit(
    X_train_ml,
    y_train.values.ravel()
)

# Predict independent test set
rf_pred = rf.predict(X_test_ml)

# Evaluate test-set performance
rf_r2 = r2_score(y_test.values.ravel(), rf_pred)

rf_rmse = np.sqrt(
    mean_squared_error(y_test.values.ravel(), rf_pred)
)

rf_mae = mean_absolute_error(
    y_test.values.ravel(), rf_pred
)

print("\nIndependent Test Set Performance:")
print(f"RF Test R²   = {rf_r2:.4f}")
print(f"RF Test RMSE = {rf_rmse:.4f}")
print(f"RF Test MAE  = {rf_mae:.4f}")


**XGBoost_Hyperparameter_Tuning_and_Evaluation**

In [ ]:

# Install XGBoost if necessary
# !pip install xgboost

import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# Initialize XGBoost
xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist",
    n_jobs=-1
)

# Train on the training set
xgb.fit(
    X_train_ml,
    y_train.values.ravel()
)

# Predict independent test set
xgb_pred = xgb.predict(X_test_ml)

# Calculate evaluation metrics
xgb_r2 = r2_score(
    y_test.values.ravel(),
    xgb_pred
)

xgb_rmse = np.sqrt(
    mean_squared_error(
        y_test.values.ravel(),
        xgb_pred
    )
)

xgb_mae = mean_absolute_error(
    y_test.values.ravel(),
    xgb_pred
)

# Display results
print("XGBoost Test Set Results")
print(f"R²   = {xgb_r2:.4f}")
print(f"RMSE = {xgb_rmse:.4f}")
print(f"MAE  = {xgb_mae:.4f}")


**SVR_Hyperparameter_Tuning_and_Evaluation**

In [ ]:

import numpy as np

from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.svm import SVR
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

# SVR hyperparameter search space
svr_grid = {
    "C": [1, 10, 50, 100, 200],
    "gamma": ["scale", 0.01, 0.001],
    "epsilon": [0.01, 0.05, 0.1]
}

# Five-fold cross-validation
cv_strategy = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Randomized hyperparameter optimization
svr_search = RandomizedSearchCV(
    estimator=SVR(kernel="rbf"),
    param_distributions=svr_grid,
    n_iter=10,
    cv=cv_strategy,
    scoring="r2",
    n_jobs=-1,
    random_state=42,
    refit=True
)

# Fit using training data only
svr_search.fit(
    X_train_ml,
    y_train.values.ravel()
)

print("Best SVR Parameters:")
print(svr_search.best_params_)

print(
    f"Best CV R² = {svr_search.best_score_:.4f}"
)

# Retrieve best model
svr = svr_search.best_estimator_

# Predict independent test set
svr_pred = svr.predict(X_test_ml)

# Evaluate performance
svr_r2 = r2_score(
    y_test.values.ravel(),
    svr_pred
)

svr_rmse = np.sqrt(
    mean_squared_error(
        y_test.values.ravel(),
        svr_pred
    )
)

svr_mae = mean_absolute_error(
    y_test.values.ravel(),
    svr_pred
)

print("\nSVR Independent Test Results")
print(f"R²   = {svr_r2:.4f}")
print(f"RMSE = {svr_rmse:.4f}")
print(f"MAE  = {svr_mae:.4f}")


**Stacked Ensemble QSAR Framework**

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)
import numpy as np

# Define stacking ensemble
stack_model = StackingRegressor(
    estimators=[
        ('svr', svr),
        ('rf', rf),
        ('xgb', xgb)
    ],
    final_estimator=LinearRegression(),
    cv=5,
    n_jobs=-1
)

# Train the stacking ensemble
stack_model.fit(
    X_train_ml,
    y_train.values.ravel()
)

# Predict on independent test data
stack_pred = stack_model.predict(X_test_ml)

# Evaluate performance
stack_r2 = r2_score(
    y_test.values.ravel(),
    stack_pred
)

stack_rmse = np.sqrt(
    mean_squared_error(
        y_test.values.ravel(),
        stack_pred
    )
)

stack_mae = mean_absolute_error(
    y_test.values.ravel(),
    stack_pred
)

print("Stacked Ensemble")
print(f"R²   = {stack_r2:.4f}")
print(f"RMSE = {stack_rmse:.4f}")
print(f"MAE  = {stack_mae:.4f}")